<a href="https://colab.research.google.com/github/arjun-1238/Machine-Learning-Models/blob/main/RandomForest_HR_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [84]:
import pandas as pd

# Load a standard HR attrition dataset directly via URL
url = "https://raw.githubusercontent.com/IBM/employee-attrition-aif360/master/data/emp_attrition.csv"
df = pd.read_csv(url)

In [85]:
df.columns

Index(['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department',
       'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount',
       'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate',
       'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction',
       'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked',
       'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating',
       'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel',
       'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
       'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager'],
      dtype='object')

In [86]:
df.isnull().sum()

,0
Age,0
Attrition,0
BusinessTravel,0
DailyRate,0
Department,0
DistanceFromHome,0
Education,0
EducationField,0
EmployeeCount,0
EmployeeNumber,0


In [87]:
from sklearn.utils import resample

y_for_balancing = df["Attrition"].map({"Yes":1,"No":0})

combined_df = df.copy()
combined_df['Attrition'] = y_for_balancing
df_majority = combined_df[combined_df['Attrition'] == 0]
df_minority = combined_df[combined_df['Attrition'] == 1]


df_majority_downsampled = resample(
    df_majority,
    replace=False,
    n_samples=len(df_minority),
    random_state=42
)

balanced_df = pd.concat([df_majority_downsampled, df_minority])
df_balanced = balanced_df.drop('Attrition', axis=1)
y_balanced = balanced_df['Attrition']

print(y_balanced.value_counts())

Attrition
0    237
1    237
Name: count, dtype: int64


In [88]:
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [89]:
df["EducationField"].value_counts()

,count
EducationField,
Life Sciences,606
Medical,464
Marketing,159
Technical Degree,132
Other,82
Human Resources,27


In [90]:

X_features = df_balanced.drop(["EmployeeCount", "StandardHours", "EmployeeNumber", "Over18"], axis=1, errors='ignore')
x_category = X_features.select_dtypes(include=["object", "category"])
print(x_category.columns)

X_encoded = pd.get_dummies(X_features, columns=x_category.columns, drop_first=True)
print(X_encoded.shape)


Index(['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole',
       'MaritalStatus', 'OverTime'],
      dtype='object')
(474, 44)


In [91]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_final = X_encoded.copy()
numeric_cols_to_scale = X_final.select_dtypes(include=["int64", "float64"]).columns.tolist()


if 'Age' in numeric_cols_to_scale:
    numeric_cols_to_scale.remove('Age')

if numeric_cols_to_scale:
    X_final[numeric_cols_to_scale] = scaler.fit_transform(X_final[numeric_cols_to_scale])

In [92]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,recall_score,precision_score,f1_score




In [93]:

x_train,x_test,y_train,y_test=train_test_split(X_final,y_balanced,test_size=0.3,random_state=42)

In [94]:
print(df.shape)
y

(1470, 35)


,Attrition
0,1
1,0
2,1
3,0
4,0
...,...
1465,0
1466,0
1467,0
1468,0


In [95]:
model=RandomForestClassifier(n_estimators=300,class_weight="balanced")
model.fit(x_train,y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=300)

In [96]:
predict=model.predict(x_test)

In [97]:
print(accuracy_score(y_test,predict))
print(recall_score(y_test,predict))
print(precision_score(y_test,predict))
print(classification_report(y_test,predict))
print(confusion_matrix(y_test,predict))

0.6993006993006993
0.7142857142857143
0.684931506849315
              precision    recall  f1-score   support

           0       0.71      0.68      0.70        73
           1       0.68      0.71      0.70        70

    accuracy                           0.70       143
   macro avg       0.70      0.70      0.70       143
weighted avg       0.70      0.70      0.70       143

[[50 23]
 [20 50]]


In [98]:

y_pred_proba = model.predict_proba(x_test)[:, 1]

print("First 10 prediction probabilities:")
print(y_pred_proba[:10])

First 10 prediction probabilities:
[0.51333333 0.61333333 0.47333333 0.69333333 0.57       0.12333333
 0.48       0.26333333 0.27       0.37333333]


In [99]:
def evaluate_model_with_threshold(y_true, y_pred_proba, threshold):

    y_pred_thresholded = (y_pred_proba >= threshold).astype(int)
    print("Accuracy", accuracy_score(y_true,y_pred_thresholded))
    print("Recall",recall_score(y_true,y_pred_thresholded))
    print("Precisiom", precision_score(y_true, y_pred_thresholded))
    print("F1-Score", f1_score(y_true, y_pred_thresholded))
    print("\nClassification Report",classification_report(y_true, y_pred_thresholded))
    print("Confusion Matri",confusion_matrix(y_true, y_pred_thresholded))

In [100]:
evaluate_model_with_threshold(y_test, y_pred_proba, 0.3)

Accuracy 0.5594405594405595
Recall 0.9571428571428572
Precisiom 0.5275590551181102
F1-Score 0.6802030456852792

Classification Report               precision    recall  f1-score   support

           0       0.81      0.18      0.29        73
           1       0.53      0.96      0.68        70

    accuracy                           0.56       143
   macro avg       0.67      0.57      0.49       143
weighted avg       0.67      0.56      0.48       143

Confusion Matri [[13 60]
 [ 3 67]]


Now, let's try a higher threshold (e.g., 0.7) to prioritize precision.

In [101]:
evaluate_model_with_threshold(y_test, y_pred_proba, 0.5)

Accuracy 0.6993006993006993
Recall 0.7142857142857143
Precisiom 0.684931506849315
F1-Score 0.6993006993006993

Classification Report               precision    recall  f1-score   support

           0       0.71      0.68      0.70        73
           1       0.68      0.71      0.70        70

    accuracy                           0.70       143
   macro avg       0.70      0.70      0.70       143
weighted avg       0.70      0.70      0.70       143

Confusion Matri [[50 23]
 [20 50]]
